# Phase 3: Evaluation & Anomaly Scoring (MLflow Engine)

---

## What This Notebook Does

This is the **final phase** of our pipeline. It takes the trained Autoencoder from Phase 2
and scores test audio files to detect anomalies. All metrics are logged to MLflow.

### Two Scoring Paradigms: Reconstruction vs. Embeddings

| Method | How It Scores | Domain Shift Resistance |
|--------|--------------|------------------------|
| **Reconstruction Error** | Passes data through full Autoencoder, measures pixel-wise copy error | **Low** — sensitive to mic position, background noise, RPM changes |
| **Isolation Forest** | Fits outlier detector on 64-dim bottleneck embeddings | **High** — embeddings capture abstract physics, not surface pixels |

### How Isolation Forest Solves Domain Shift

DCASE 2024 test machines are recorded under different conditions than training.
Reconstruction Error sees surface changes and false-alarms. But bottleneck
embeddings capture the **underlying acoustic physics**:

```
Normal fan (lab)     → Encoder → [0.3, -0.1, 0.8, ..., 0.2]  ← cluster center
Normal fan (factory) → Encoder → [0.4, -0.2, 0.7, ..., 0.3]  ← nearby (same physics)
Broken fan (factory) → Encoder → [2.1,  1.5, -3.0, ..., 4.7] ← far away (anomaly!)
```

The Isolation Forest draws a **boundary** around the normal cluster. Domain-shifted
but healthy sounds stay inside; truly broken sounds fly far outside.

### Partial AUC (pAUC): The Factory-Friendly Metric

Standard AUC treats all false alarm rates equally. But in a factory, a 50% false alarm
rate is catastrophic — operators would ignore the system. **pAUC** measures AUC only
in the low false-positive region (FPR ≤ 10%), penalizing false alarms heavily.

---

## 1. Control Panel

In [31]:
EVAL_PARAMS = {
    'SCORING_METHOD': 'IsolationForest',
    'PATCH_WIDTH':    32,
    'PATCH_STRIDE':   16,
    'SAMPLE_RATE':    16000,
    'N_FFT':          1024,
    'HOP_LENGTH':     512,
    'N_MELS':         128,
    'IF_CONTAMINATION': 0.05,
    'GMM_COMPONENTS':   5,
}

print("EVAL CONTROL PANEL")
print("=" * 50)
for k, v in EVAL_PARAMS.items():
    print(f"  {k:<20s} : {v}")

EVAL CONTROL PANEL
  SCORING_METHOD       : IsolationForest
  PATCH_WIDTH          : 32
  PATCH_STRIDE         : 16
  SAMPLE_RATE          : 16000
  N_FFT                : 1024
  HOP_LENGTH           : 512
  N_MELS               : 128
  IF_CONTAMINATION     : 0.05
  GMM_COMPONENTS       : 5


---

## 2. Library Imports

| Library | Purpose |
|---------|--------|
| **`librosa`** | Loads raw `.wav` test files and computes spectrograms matching Phase 1 settings. |
| **`sklearn.ensemble.IsolationForest`** | Tree-based outlier detector fitted on bottleneck embeddings. |
| **`sklearn.mixture.GaussianMixture`** | Probabilistic density estimator — low-probability = anomaly. |
| **`sklearn.metrics.roc_auc_score`** | Computes the Area Under the ROC Curve. |

In [32]:
import os, json, glob
import numpy as np
import torch
import torch.nn as nn
import librosa
import joblib
import mlflow
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.ensemble import IsolationForest
from sklearn.mixture import GaussianMixture
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

Device: cuda


---

## 3. Architecture (must match training exactly)

In [33]:
class CNNAutoencoder(nn.Module):
    def __init__(self, input_height, patch_width, bottleneck_dim, dropout_rate=0.2):
        super().__init__()
        self.encoder = nn.Sequential(
            self._cb(1, 32, dropout_rate), self._cb(32, 64, dropout_rate),
            self._cb(64, 128, dropout_rate), self._cb(128, 128, dropout_rate))
        h, w = input_height // 16, patch_width // 16
        self.flat_size, self.h_enc, self.w_enc = 128 * h * w, h, w
        self.flatten = nn.Flatten()
        self.bottleneck_encode = nn.Linear(self.flat_size, bottleneck_dim)
        self.bottleneck_decode = nn.Linear(bottleneck_dim, self.flat_size)
        self.decoder = nn.Sequential(
            self._db(128, 128), self._db(128, 64), self._db(64, 32),
            nn.ConvTranspose2d(32, 1, 3, 2, 1, output_padding=1), nn.Sigmoid())

    def _cb(self, ic, oc, dr):
        return nn.Sequential(nn.Conv2d(ic, oc, 3, 1, 1), nn.BatchNorm2d(oc),
                             nn.LeakyReLU(0.2), nn.MaxPool2d(2, 2), nn.Dropout2d(dr))
    def _db(self, ic, oc):
        return nn.Sequential(nn.ConvTranspose2d(ic, oc, 3, 2, 1, output_padding=1),
                             nn.BatchNorm2d(oc), nn.LeakyReLU(0.2))
    def forward(self, x):
        e = self.encoder(x)
        z = self.bottleneck_encode(self.flatten(e))
        d = self.bottleneck_decode(z).view(-1, 128, self.h_enc, self.w_enc)
        return self.decoder(d)
    def encode(self, x):
        return self.bottleneck_encode(self.flatten(self.encoder(x)))

print("Architecture defined.")

Architecture defined.


---

## 4. Helper Functions

In [34]:
def extract_patches(data, width, stride):
    patches = []
    for spec in data:
        for s in range(0, spec.shape[1] - width + 1, stride):
            patches.append(spec[:, s:s + width])
    return np.array(patches)

def compute_pauc(y_true, y_score, max_fpr=0.1):
    fpr, tpr, _ = roc_curve(y_true, y_score)
    stop = np.searchsorted(fpr, max_fpr, side='right')
    if stop == 0: return 0.0
    fpr_t, tpr_t = fpr[:stop], tpr[:stop]
    if fpr_t[-1] < max_fpr:
        tpr_t = np.append(tpr_t, np.interp(max_fpr, fpr, tpr))
        fpr_t = np.append(fpr_t, max_fpr)
    return np.trapz(tpr_t, fpr_t) / max_fpr

def find_latest_run_folder(runs_dir):
    folders = sorted([d for d in os.listdir(runs_dir)
                      if os.path.isdir(os.path.join(runs_dir, d)) and d.startswith('run_')])
    if folders: return os.path.join(runs_dir, folders[-1])
    fb = os.path.join(runs_dir, "eval_results")
    os.makedirs(fb, exist_ok=True)
    return fb

print("Helpers defined.")

Helpers defined.


---

## 5. Evaluation Pipeline

The **Dimension Crop Fix** is applied to test spectrograms so the model
receives the same height it was trained on (e.g., 513 → 512).

In [35]:
# --- Local paths (self-contained briefcase) ---
RAW_BASE   = os.path.join(".", "data", "raw")
PROC_BASE  = os.path.join(".", "data", "processed")
MODELS_DIR = os.path.join(".", "experiments", "models")
RUNS_DIR   = os.path.join(".", "experiments", "runs")

run_folder = find_latest_run_folder(RUNS_DIR)
print(f"Saving visuals to: {run_folder}")

machines = sorted([d for d in os.listdir(PROC_BASE) if os.path.isdir(os.path.join(PROC_BASE, d))])
PW, PS, SR = EVAL_PARAMS['PATCH_WIDTH'], EVAL_PARAMS['PATCH_STRIDE'], EVAL_PARAMS['SAMPLE_RATE']
SCORING = EVAL_PARAMS['SCORING_METHOD']
criterion_pixel = nn.L1Loss(reduction='none')
final_metrics = {}

mlflow.set_experiment("AcousticAnomalyDetection")

for machine in machines:
    print(f"\n{'='*70}\nEVALUATING: {machine} ({SCORING})\n{'='*70}")

    model_path  = os.path.join(MODELS_DIR, f"best_model_{machine}.pth")
    scaler_path = os.path.join(PROC_BASE, machine, "scaler.save")
    if not os.path.exists(model_path) or not os.path.exists(scaler_path):
        print("  Missing model/scaler."); continue

    meta_path = os.path.join(PROC_BASE, machine, "prep_meta.json")
    if os.path.exists(meta_path):
        with open(meta_path) as f: pm = json.load(f)
        input_height = pm['SPEC_HEIGHT']
        spec_type = pm.get('SPECTROGRAM_TYPE', 'Mel')
    else:
        input_height, spec_type = 128, 'Mel'

    # ===== DIMENSION CROP FIX =====
    model_height = (input_height // 16) * 16
    # ==============================

    state = torch.load(model_path, map_location=device)
    bn_dim = state['bottleneck_encode.weight'].shape[0]
    model = CNNAutoencoder(model_height, PW, bn_dim).to(device)
    model.load_state_dict(state)
    model.eval()
    scaler = joblib.load(scaler_path)

    # Fit embedding scorer if needed
    embedding_scorer = None
    if SCORING in ('IsolationForest', 'GMM'):
        print("  Fitting embedding scorer...")
        X_tr = np.load(os.path.join(PROC_BASE, machine, "X_train.npy"))
        X_tr = X_tr[:, :model_height, :]  # Crop fix
        tr_patches = extract_patches(X_tr, PW, PS)
        tr_t = torch.tensor(tr_patches, dtype=torch.float32).unsqueeze(1).to(device)
        embeds = []
        with torch.no_grad():
            for i in range(0, len(tr_t), 256):
                embeds.append(model.encode(tr_t[i:i+256]).cpu().numpy())
        embeds = np.concatenate(embeds)
        if SCORING == 'IsolationForest':
            embedding_scorer = IsolationForest(
                contamination=EVAL_PARAMS['IF_CONTAMINATION'], random_state=42, n_jobs=-1)
        else:
            embedding_scorer = GaussianMixture(
                n_components=EVAL_PARAMS['GMM_COMPONENTS'], random_state=42)
        embedding_scorer.fit(embeds)
        print(f"  {SCORING} fitted on {embeds.shape}")

    test_files = sorted(glob.glob(os.path.join(RAW_BASE, machine, "test", "*.wav")))
    if not test_files: print("  No test files."); continue

    labels, scores, example_anomaly = [], [], None

    for fpath in test_files:
        fname = os.path.basename(fpath).lower()
        is_anomaly = 1 if 'anomaly' in fname else 0
        try: y, sr = librosa.load(fpath, sr=SR)
        except: continue

        if spec_type == 'Linear':
            stft = np.abs(librosa.stft(y, n_fft=EVAL_PARAMS['N_FFT'], hop_length=EVAL_PARAMS['HOP_LENGTH']))
            spec_db = librosa.amplitude_to_db(stft, ref=np.max)
        else:
            mel = librosa.feature.melspectrogram(
                y=y, sr=sr, n_fft=EVAL_PARAMS['N_FFT'],
                hop_length=EVAL_PARAMS['HOP_LENGTH'], n_mels=EVAL_PARAMS['N_MELS'])
            spec_db = librosa.power_to_db(mel, ref=np.max)

        H, W = spec_db.shape
        scaled = scaler.transform(spec_db.reshape(1, H * W)).reshape(H, W)

        # ===== DIMENSION CROP FIX =====
        scaled = scaled[:model_height, :]
        # ==============================

        fp_list = [scaled[:, s:s+PW] for s in range(0, scaled.shape[1] - PW + 1, PS)]
        if not fp_list: continue
        batch_t = torch.tensor(np.array(fp_list), dtype=torch.float32).unsqueeze(1).to(device)

        with torch.no_grad():
            if SCORING == 'Reconstruction':
                out = model(batch_t)
                errs = criterion_pixel(out, batch_t)
                pe = errs.mean(dim=(1, 2, 3)).cpu().numpy()
                file_score = float(np.max(pe))
                if is_anomaly and example_anomaly is None:
                    wi = np.argmax(pe)
                    example_anomaly = {'input': batch_t[wi,0].cpu().numpy(),
                        'recon': out[wi,0].cpu().numpy(), 'error': errs[wi,0].cpu().numpy(),
                        'machine': machine}
            else:
                z = model.encode(batch_t).cpu().numpy()
                if SCORING == 'IsolationForest':
                    ps = -embedding_scorer.decision_function(z)
                else:
                    ps = -embedding_scorer.score_samples(z)
                file_score = float(np.max(ps))

        labels.append(is_anomaly); scores.append(file_score)

    if len(labels) > 0 and len(set(labels)) > 1:
        auc = roc_auc_score(labels, scores)
        pauc = compute_pauc(labels, scores)
        verdict = "EXCELLENT" if auc >= 0.80 else "GOOD" if auc >= 0.70 else "Poor"
        final_metrics[machine] = {'auc': auc, 'pauc': pauc,
            'n_normal': labels.count(0), 'n_anomaly': labels.count(1), 'verdict': verdict}
        print(f"  AUC: {auc:.4f} | pAUC: {pauc:.4f} | {verdict}")

        ns = [s for l, s in zip(labels, scores) if l == 0]
        an = [s for l, s in zip(labels, scores) if l == 1]
        fig, ax = plt.subplots(figsize=(8, 4))
        ax.hist(ns, bins=40, alpha=0.6, label='Normal', color='steelblue')
        ax.hist(an, bins=40, alpha=0.6, label='Anomaly', color='crimson')
        ax.set_title(f'{machine} — {SCORING}'); ax.legend()
        fig.savefig(os.path.join(run_folder, f"score_dist_{machine}.png"), dpi=150, bbox_inches='tight')
        plt.close(fig)

    if example_anomaly and example_anomaly['machine'] == machine:
        fig, axes = plt.subplots(1, 3, figsize=(15, 4))
        for ax, img, t in zip(axes,
            [example_anomaly['input'], example_anomaly['recon'], example_anomaly['error']],
            ['Original', 'Reconstruction', 'Error']):
            im = ax.imshow(img, aspect='auto', origin='lower', cmap='magma')
            ax.set_title(t); plt.colorbar(im, ax=ax)
        fig.savefig(os.path.join(run_folder, f"recon_{machine}.png"), dpi=150, bbox_inches='tight')
        plt.close(fig)

print(f"\nVisuals saved to: {run_folder}")

Saving visuals to: .\experiments\runs\run_20260424_001819

EVALUATING: ToyCar (IsolationForest)
  Fitting embedding scorer...


C:\Users\MA21\AppData\Local\Temp\ipykernel_32304\3291543882.py:38: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state = torch.load(model_path, map_location=device)


  IsolationForest fitted on (18700, 32)


C:\Users\MA21\AppData\Local\Temp\ipykernel_32304\2286005354.py:16: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return np.trapz(tpr_t, fpr_t) / max_fpr
C:\Users\MA21\AppData\Local\Temp\ipykernel_32304\3291543882.py:38: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start 

  AUC: 0.4798 | pAUC: 0.0400 | Poor

EVALUATING: ToyTrain (IsolationForest)
  Fitting embedding scorer...
  IsolationForest fitted on (18700, 32)


...

C:\Users\MA21\AppData\Local\Temp\ipykernel_32304\3291543882.py:38: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state = torch.load(model_path, map_location=device)


  IsolationForest fitted on (15300, 32)


C:\Users\MA21\AppData\Local\Temp\ipykernel_32304\2286005354.py:16: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return np.trapz(tpr_t, fpr_t) / max_fpr
C:\Users\MA21\AppData\Local\Temp\ipykernel_32304\3291543882.py:38: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start 

  AUC: 0.4586 | pAUC: 0.0320 | Poor

EVALUATING: valve (IsolationForest)
  Fitting embedding scorer...
  IsolationForest fitted on (15300, 32)
  AUC: 0.4436 | pAUC: 0.0540 | Poor

Visuals saved to: .\experiments\runs\run_20260424_001819


C:\Users\MA21\AppData\Local\Temp\ipykernel_32304\2286005354.py:16: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return np.trapz(tpr_t, fpr_t) / max_fpr


---

## 6. Final Report & MLflow Logging

In [36]:
print(f"\n{'='*72}")
print(f"ANOMALY DETECTION RESULTS — {SCORING}")
print(f"{'='*72}")
print(f"{'Machine':<12} | {'AUC':>8} | {'pAUC':>8} | {'Normal':>6} | {'Anom':>5} | {'Verdict':>10}")
print("-" * 72)

auc_list, pauc_list = [], []
for m, r in final_metrics.items():
    print(f"{m:<12} | {r['auc']:8.4f} | {r['pauc']:8.4f} | "
          f"{r['n_normal']:6d} | {r['n_anomaly']:5d} | {r['verdict']:>10}")
    auc_list.append(r['auc'])
    pauc_list.append(r['pauc'])

print("-" * 72)
if auc_list:
    avg_auc = np.mean(auc_list)
    avg_pauc = np.mean(pauc_list)
    print(f"{'AVERAGE':<12} | {avg_auc:8.4f} | {avg_pauc:8.4f} |")
print("=" * 72)

with mlflow.start_run(run_name=f"eval_{SCORING}"):
    mlflow.log_params({f"eval_{k}": v for k, v in EVAL_PARAMS.items()})
    for m, r in final_metrics.items():
        mlflow.log_metric(f"auc_{m}", r['auc'])
        mlflow.log_metric(f"pauc_{m}", r['pauc'])
    if auc_list:
        mlflow.log_metric("avg_auc", avg_auc)
        mlflow.log_metric("avg_pauc", avg_pauc)
    for f in os.listdir(run_folder):
        fp = os.path.join(run_folder, f)
        if os.path.isfile(fp): mlflow.log_artifact(fp)

print("\n✓ Results logged to MLflow. Run `mlflow ui` to view the dashboard.")


ANOMALY DETECTION RESULTS — IsolationForest
Machine      |      AUC |     pAUC | Normal |  Anom |    Verdict
------------------------------------------------------------------------
ToyCar       |   0.4798 |   0.0400 |    100 |   100 |       Poor
ToyTrain     |   0.5762 |   0.1010 |    100 |   100 |       Poor
bearing      |   0.5234 |   0.0210 |    100 |   100 |       Poor
fan          |   0.5223 |   0.0300 |    100 |   100 |       Poor
gearbox      |   0.4716 |   0.0440 |    100 |   100 |       Poor
slider       |   0.4586 |   0.0320 |    100 |   100 |       Poor
valve        |   0.4436 |   0.0540 |    100 |   100 |       Poor
------------------------------------------------------------------------
AVERAGE      |   0.4965 |   0.0460 |

✓ Results logged to MLflow. Run `mlflow ui` to view the dashboard.


---

## Summary & Next Steps

1. ✅ Loaded trained models and scalers
2. ✅ Applied the Dimension Crop Fix to test spectrograms
3. ✅ Scored with Reconstruction Error or Isolation Forest
4. ✅ Computed AUC / pAUC metrics
5. ✅ Logged everything to MLflow

**Next:** Run `run_grid_search.py` to test all 24 hyperparameter combinations overnight.